In [ ]:
%pip install datasets evaluate
%pip install transformers[torch]
%pip install accelerate
%pip install seqeval

In [2]:
from pathlib import Path
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import numpy as np
import evaluate

In [3]:
encoder_id = "bert-base-cased"
# Use the fast tokenizer version, which supports word_ids()
tokenizer = BertTokenizerFast.from_pretrained(encoder_id)

In [4]:
def tokenize_and_align_labels(examples):
    # Use padding='max_length' to ensure consistent sequence lengths
    tokenized_inputs = tokenizer(
        examples["tokens"],
        padding="max_length",
        truncation=True,
        max_length=256,  # model_max_length from original hyperparameters
        is_split_into_words=True,
    )
    
    labels_aligned = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels_aligned.append(label_ids)
    
    tokenized_inputs["labels"] = labels_aligned
    return tokenized_inputs


# Load dataset and adjust columns
dataset_id = "DFKI-SLT/few-nerd"
dataset_name = "FewNERD"
dataset = load_dataset(dataset_id, "supervised")
dataset = dataset.remove_columns("ner_tags")
dataset = dataset.rename_column("fine_ner_tags", "ner_tags")
labels = dataset["train"].features["ner_tags"].feature.names

# Initialize model for token classification with BERT.
model = BertForTokenClassification.from_pretrained(encoder_id, num_labels=len(labels))

# Tokenize the datasets using the custom function.
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/37648 [00:00<?, ? examples/s]

In [7]:
# Prepare training arguments.
model_id = f"bert-token-classification-{encoder_id}-fewnerd-fine-super"
output_dir = Path("models") / model_id
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=5e-5,
    per_device_train_batch_size=64,  # Increased batch size for training
    per_device_eval_batch_size=64,   # Increased batch size for evaluation
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=True,  # Replace with fp16 if your hardware does not support bf16.
    logging_first_step=True,
    logging_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=3000,
    save_total_limit=2,
    dataloader_num_workers=2,
)

# Load evaluation metric.
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels_batch = p
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = [
        [labels[pred] for pred, label in zip(prediction, label_seq) if label != -100]
        for prediction, label_seq in zip(predictions, labels_batch)
    ]
    true_labels = [
        [labels[label] for pred, label in zip(prediction, label_seq) if label != -100]
        for prediction, label_seq in zip(predictions, labels_batch)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
# Use a dedicated data collator to handle padding of both inputs and labels.
data_collator = DataCollatorForTokenClassification(tokenizer)

# Initialize Trainer without directly passing tokenizer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model.
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss


In [ ]:
# Evaluate on test set and save metrics.
test_metrics = trainer.evaluate(tokenized_datasets["test"], metric_key_prefix="test")
trainer.save_metrics("test", test_metrics)

# Save final model checkpoint.
trainer.save_model(output_dir / "checkpoint-final")